# YOLOv8 鹿偵測 — Colab (免費 GPU)
先按 **執行階段 → 變更執行階段類型 → GPU**。

資料集有兩條路：**A. Open Images(免 API key，已驗證可跑)**；**B. Roboflow(需免費 API key)**。
選一條執行即可，之後的訓練/評估共用。

## 1. 安裝與確認 GPU

In [ ]:
!pip -q install ultralytics fiftyone roboflow huggingface_hub
import torch; print('CUDA:', torch.cuda.is_available())
!nvidia-smi -L

## 2A. 資料集 — Open Images「Deer」(免 API key，推薦先用這條)
自動下載含 bounding box 的真實鹿圖並轉成 YOLO 格式。本專案的成果圖就是用這條路產生的。

In [ ]:
import fiftyone as fo, fiftyone.zoo as foz
from fiftyone import ViewField as F
OUT='deer_yolo'
tr=foz.load_zoo_dataset('open-images-v7',split='train',label_types=['detections'],
      classes=['Deer'],max_samples=300)
va=foz.load_zoo_dataset('open-images-v7',split='validation',label_types=['detections'],
      classes=['Deer'],max_samples=80)
for ds,sp in [(tr,'train'),(va,'val')]:
    ds.filter_labels('ground_truth',F('label')=='Deer').export(
        export_dir=OUT,dataset_type=fo.types.YOLOv5Dataset,
        label_field='ground_truth',split=sp,classes=['Deer'])
data_yaml=OUT+'/dataset.yaml'; print('data.yaml =>',data_yaml)

## 2B. (替代) 資料集 — Roboflow
若想用 Roboflow：免費註冊取得 API key，到 universe.roboflow.com 選一個 deer 偵測資料集，
換掉 workspace/project/version。**跑了 2A 就不用跑 2B。**

In [ ]:
# from roboflow import Roboflow
# rf = Roboflow(api_key='貼上你的_API_KEY')
# ds = rf.workspace('university-itgbn').project('deer-ipsw2').version(2).download('yolov8')
# data_yaml = ds.location + '/data.yaml'

## 3. 訓練
有 GPU 時 epochs 建議 50~100、imgsz 640，信心分數會校準得更好。

In [ ]:
from ultralytics import YOLO
# 預訓練權重：GitHub 若被擋，改從 HuggingFace 取得
try:
    from huggingface_hub import hf_hub_download
    w = hf_hub_download('Ultralytics/YOLOv8','yolov8n.pt')
except Exception:
    w = 'yolov8n.pt'
model = YOLO(w)
model.train(data=data_yaml, epochs=60, imgsz=640, batch=16, patience=15,
            seed=42, name='deer_yolov8n',
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=5, translate=0.1,
            scale=0.5, fliplr=0.5, mosaic=1.0, mixup=0.1)

## 4. 評估 (mAP)

In [ ]:
m = model.val()
print('mAP50:', round(float(m.box.map50),3), '| mAP50-95:', round(float(m.box.map),3),
      '| Recall:', round(float(m.box.mr),3))
from IPython.display import Image, display
display(Image('runs/detect/deer_yolov8n/results.png'))
display(Image('runs/detect/deer_yolov8n/confusion_matrix.png'))

## 5. 推論與視覺化

In [ ]:
import glob
val_imgs = glob.glob('deer_yolo/images/val/*.jpg')[:6]
res = model.predict(val_imgs, conf=0.25, save=True, max_det=5)
for p in glob.glob('runs/detect/predict*/*.jpg')[:6]:
    display(Image(p))

## 6. 下載訓練好的權重

In [ ]:
from google.colab import files
files.download('runs/detect/deer_yolov8n/weights/best.pt')